# RAG Support Chatbot — Milestone 3 (VS Code / local version)
## Advanced Techniques & Deployment

This notebook **tests** the RAG chain interactively.
The actual production code lives in:
- `src/rag_chain.py` — core RAG logic (retrieval + generation)
- `src/api.py`       — FastAPI REST server wrapping the chain

**Pipeline per query:**
```
user question
  → embed with sentence-transformers (all-MiniLM-L6-v2)
  → hybrid retrieve top-3 from FAISS index
  → build prompt: system message + context docs + question
  → generate answer with flan-t5-base (local CPU)
  → return answer + sources
```

**Steps:**
```
Step 1 → Install new dependencies
Step 2 → Load all models (embedding + FAISS + LLM)
Step 3 → Test the RAG chain interactively
Step 4 → Evaluate answer quality
Step 5 → Test the REST API
Step 6 → Security notes
```

## Step 1 — Install new dependencies

In [1]:
# Run this once in your activated venv terminal:
#   pip install transformers torch fastapi uvicorn[standard] pydantic httpx
#
# torch is needed by transformers for flan-t5 inference on CPU.
# httpx is needed to test the API from inside the notebook.
#
# Uncomment to install from inside the notebook:
# %pip install transformers torch fastapi uvicorn[standard] pydantic httpx

In [2]:
print("Start")

Start


## Step 2 — Load all models

In [ ]:
import importlib
import os
import sys

# Make sure Python can find the src/ package
PROJECT_ROOT = os.path.abspath('..')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import src.rag_chain as rag_chain

# Reload the module to avoid using an older copy already cached in the notebook kernel.
rag_chain = importlib.reload(rag_chain)

load_all = rag_chain.load_all
rebuild_index = rag_chain.rebuild_index

load_all()
rebuild_index()   # takes ~5-10 min, only needed once

ImportError: cannot import name 'rebuild_index' from 'src.rag_chain' (c:\Users\lojyn\OneDrive\Documents\GitHub\NHA-4-231\src\rag_chain.py)

In [ ]:
import importlib
import os
import sys

# Make sure Python can find the src/ package
PROJECT_ROOT = os.path.abspath('..')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import src.rag_chain as rag_chain

# Reload the module to avoid using an older copy already cached in the notebook kernel.
rag_chain = importlib.reload(rag_chain)

load_all = rag_chain.load_all
ask = rag_chain.ask
search = rag_chain.search

# Load everything into memory.
# First run: downloads flan-t5-base (~250MB) from Hugging Face.
# Subsequent runs: loads from local cache — much faster.
load_all()
print('All models loaded and ready.')

[RAG] Loading embedding model...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2770.52it/s]


[RAG] Loading FAISS index...
[RAG] Loading train lookup table...
[RAG] Loading BM25 corpus...
[RAG] Loading LLM (google/flan-t5-base)... (downloads ~250MB first time)


c:\Users\lojyn\OneDrive\Documents\GitHub\NHA-4-231\venv\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\lojyn\.cache\huggingface\hub\models--google--flan-t5-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 282/282 [00:00<00:00, 5165.56it/s]
[transformers] The tied 

[RAG] All models loaded. Ready.

All models loaded and ready.


## Step 3 — Test the RAG chain interactively

In [10]:
def print_result(result: dict) -> None:
    """Pretty-print a RAG chain result dict."""
    print(f"QUERY    : {result['query']}")
    print(f"RETRIEVAL: {result['retrieval']}")
    print(f"\nANSWER:\n{result['answer']}")
    print(f"\nSOURCES USED ({len(result['sources'])} docs):")
    for i, src in enumerate(result['sources'], 1):
        print(f"  #{i} [{src['category']} -> {src['intent']}]  score={src['score']:.4f}")
        print(f"      Q: {src['instruction']}")
        print(f"      A: {src['response'][:100]}...")
    print('='*65)

In [11]:
# --- Test 1: Order cancellation ---
result = ask("I want to cancel my order")
print_result(result)

QUERY    : I want to cancel my order
RETRIEVAL: hybrid

ANSWER:
Order History page. 3. Locate your order: Look for the order labeled with the order number your order number and click on it. 4. Initiate cancellation: Within the order details page, you should find an option to "Cancel Order." Please select this option. 5. Confirm cancellation: The system may prompt you to confirm the cancellation.

SOURCES USED (3 docs):
  #1 [ORDER -> cancel_order]  score=0.8720
      Q: I do not want my last bloody item, cancel order your order number
      A: I'm picking up what you're putting down, your request to cancel your order with the order number you...
  #2 [ORDER -> cancel_order]  score=0.8686
      Q: cancel order
      A: I've realized that you're seeking assistance in canceling an order. I'll be happy to guide you throu...
  #3 [ORDER -> cancel_order]  score=0.8636
      Q: I do not want this item, I try to cancel order your order number
      A: I pick up what you're putting down, your d

In [12]:
# --- Test 2: Missing package ---
result = ask("My package hasn't arrived and it's been 2 weeks")
print_result(result)

QUERY    : My package hasn't arrived and it's been 2 weeks
RETRIEVAL: hybrid

ANSWER:
We understand your desire to track the arrival of your package and would be happy to assist you. To check the estimated arrival time, you can visit our website and log in to your account. Once you're logged in, navigate to the 'Order History' or 'Track Your Order' section, where you can find detailed information about the status of your parcel and its expected delivery date.

SOURCES USED (3 docs):
  #1 [DELIVERY -> delivery_period]  score=0.6814
      Q: I need to see when will my package arrive, how do I do it?
      A: We understand your desire to track the arrival of your package and would be happy to assist you. To ...
  #2 [DELIVERY -> delivery_period]  score=0.6413
      Q: where can I check when my package is going to arrive?
      A: We completely understand your eagerness to track the delivery status of your package. To check the e...
  #3 [DELIVERY -> delivery_period]  score=0.6335
      Q:

In [13]:
# --- Test 3: Password reset ---
result = ask("I forgot my password and I cannot log into my account")
print_result(result)

QUERY    : I forgot my password and I cannot log into my account
RETRIEVAL: hybrid

ANSWER:
reach out to me. I'm here to assist you throughout the entire process until you regain access to your account.

SOURCES USED (3 docs):
  #1 [ACCOUNT -> recover_password]  score=0.8123
      Q: i need help to reset the pass of my account
      A: I'm sorry to hear that you're having trouble resetting the password for your account. I'm here to he...
  #2 [ACCOUNT -> recover_password]  score=0.8109
      Q: I forgot the fucking pass of my account, I need to reset it
      A: I've grasped that you're frustrated and concerned about forgetting your account password. Please don...
  #3 [ACCOUNT -> recover_password]  score=0.8092
      Q: I want assistance retrieving the password of my account
      A: I'll make it happen! I understand how crucial it is for you to retrieve the password of your account...


In [14]:
# --- Test 4: Double charge ---
result = ask("I was charged twice for the same order")
print_result(result)

QUERY    : I was charged twice for the same order
RETRIEVAL: hybrid

ANSWER:
1. **Product/Service Defect:** If you receive a product or service that is faulty, defective, or significantly different from what was described, you are eligible for a reimbursement. 2.

SOURCES USED (3 docs):
  #1 [REFUND -> check_refund_policy]  score=0.6007
      Q: I have to check in which cases can I ask to be reimbursed
      A: Thank you for reaching out! I'm here to provide you with a comprehensive understanding of the cases ...
  #2 [REFUND -> check_refund_policy]  score=0.5859
      Q: i have got to see in which cases can i request my money back
      A: Definitely! I understand your need for clarity on the situations in which you can request a refund. ...
  #3 [REFUND -> check_refund_policy]  score=0.5854
      Q: I need to see in which cases can I ask for reimbursements
      A: Of course! I completely understand your need for clarity regarding the situations in which you can a...


In [15]:
# --- Test 5: Defective product return ---
result = ask("The product I received is broken, how do I return it?")
print_result(result)

QUERY    : The product I received is broken, how do I return it?
RETRIEVAL: hybrid

ANSWER:
I recommend reaching out to our customer service team. They will assist you in updating your order and ensure that the additional product is included

SOURCES USED (3 docs):
  #1 [REFUND -> get_refund]  score=0.6133
      Q: I am trying to obtain a reimbursement
      A: I'm clearly cognizant your desire to obtain a reimbursement. It's frustrating to have paid for a pro...
  #2 [ORDER -> place_order]  score=0.5591
      Q: i do not know how i can acquire a product
      A: Glad you contacted to us. I understand that you're uncertain about the process of acquiring a produc...
  #3 [ORDER -> change_order]  score=0.5435
      Q: I missed a product in order your order number, how can I add it?
      A: Thank you for informing us about the missing product in order number your order number. We apologize...


In [16]:
# --- Test 6: Noisy/informal query (simulates real customer typing) ---
result = ask("whr is my ordr?? its been forever")
print_result(result)

QUERY    : whr is my ordr?? its been forever
RETRIEVAL: hybrid

ANSWER:
My ordr is the name of a person who has been a customer for a long time.

SOURCES USED (3 docs):
  #1 [DELIVERY -> delivery_period]  score=0.5171
      Q: can i check when my package is gonna arrive
      A: We completely understand your eagerness to track the arrival of your package and determine its estim...
  #2 [DELIVERY -> delivery_period]  score=0.4630
      Q: I need help seeing how long it takes for my item to arrive
      A: We understand your curiosity about the estimated delivery time of your item. To provide you with acc...
  #3 [REFUND -> track_refund]  score=0.4561
      Q: I expect a refund of informationinformation, has it been processed?
      A: Thank you for bringing up your expected refund of informationinformation. I understand how important...


## Step 4 — Evaluate answer quality

We compare the LLM-generated answer against the gold response from the dataset.
A high ROUGE score means the generated answer closely mirrors the expected response.

In [17]:
import pandas as pd
import numpy as np
import nltk
from tqdm.auto import tqdm
from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

rouge  = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
smooth = SmoothingFunction().method1

test_df = pd.read_csv('../data/test_df.csv')

EVAL_SAMPLE = 50   # keep small — each row calls the LLM which is slow on CPU
sample = test_df.sample(EVAL_SAMPLE, random_state=42).reset_index(drop=True)

bleu_scores, r1, r2, rl = [], [], [], []

print(f'Evaluating {EVAL_SAMPLE} test queries end-to-end (retrieve + generate)...')
print('This takes a few minutes on CPU — each query runs the full LLM pipeline.\n')

for _, row in tqdm(sample.iterrows(), total=EVAL_SAMPLE):
    result      = ask(row['instruction_clean'], top_k=3)
    gold        = str(row['response_clean'])
    generated   = result['answer']

    # BLEU
    ref = nltk.word_tokenize(gold.lower())
    hyp = nltk.word_tokenize(generated.lower())
    bleu_scores.append(sentence_bleu([ref], hyp, smoothing_function=smooth))

    # ROUGE
    rs = rouge.score(gold, generated)
    r1.append(rs['rouge1'].fmeasure)
    r2.append(rs['rouge2'].fmeasure)
    rl.append(rs['rougeL'].fmeasure)

print('\n' + '='*50)
print('END-TO-END EVALUATION RESULTS (RAG chain)')
print('='*50)
print(f'  Sample size : {EVAL_SAMPLE}')
print(f'  Avg BLEU    : {np.mean(bleu_scores):.4f}')
print(f'  Avg ROUGE-1 : {np.mean(r1):.4f}')
print(f'  Avg ROUGE-2 : {np.mean(r2):.4f}')
print(f'  Avg ROUGE-L : {np.mean(rl):.4f}')
print()
print('Score guide for this task:')
print('  ROUGE-1 > 0.40 = good retrieval   ROUGE-L > 0.35 = coherent generation')

Evaluating 50 test queries end-to-end (retrieve + generate)...
This takes a few minutes on CPU — each query runs the full LLM pipeline.



100%|██████████| 50/50 [13:44<00:00, 16.49s/it]


END-TO-END EVALUATION RESULTS (RAG chain)
  Sample size : 50
  Avg BLEU    : 0.1022
  Avg ROUGE-1 : 0.3529
  Avg ROUGE-2 : 0.1768
  Avg ROUGE-L : 0.2538

Score guide for this task:
  ROUGE-1 > 0.40 = good retrieval   ROUGE-L > 0.35 = coherent generation


## Step 5 — Test the REST API

**Before running this step**, start the API server in a separate VS Code terminal:

```bash
# Make sure venv is active, then from the project root:
uvicorn src.api:app --reload --port 8000
```

Wait until you see:
```
[API] Ready to serve requests.
INFO:     Application startup complete.
```

Then run the cells below.

In [ ]:
import httpx
import json

BASE_URL = "http://localhost:8000"

# --- Health check ---
resp = httpx.get(f"{BASE_URL}/health")
print("GET /health")
print(json.dumps(resp.json(), indent=2))

In [ ]:
# --- POST /ask ---
payload = {
    "question"  : "I want to cancel my order",
    "top_k"     : 3,
    "use_hybrid": True
}
resp = httpx.post(f"{BASE_URL}/ask", json=payload, timeout=60)
data = resp.json()

print("POST /ask")
print(f"  Query  : {data['query']}")
print(f"  Answer : {data['answer']}")
print(f"  Sources: {len(data['sources'])} docs retrieved")
for src in data['sources']:
    print(f"    [{src['intent']}]  score={src['score']:.4f}")

In [ ]:
# --- GET /search (retrieval only, no generation) ---
resp = httpx.get(f"{BASE_URL}/search", params={"query": "track my delivery", "top_k": 3})
print("GET /search?query=track my delivery&top_k=3")
for doc in resp.json():
    print(f"  [{doc['intent']}]  score={doc['score']:.4f}  ->  {doc['instruction']}")

In [ ]:
# --- Swagger UI shortcut ---
# You can also test the API interactively in your browser at:
print("Interactive API docs (Swagger UI):")
print(f"  {BASE_URL}/docs")
print()
print("Raw OpenAPI schema:")
print(f"  {BASE_URL}/openapi.json")

## Step 6 — Security notes

Your Milestone 3 requirements include: *'Secure endpoints with Azure AD or API keys'*.
Since we're running locally (no Azure yet), here is how security is handled:

**Current state (local dev):**
- CORS is open (`allow_origins=["*"]`) — fine locally, must be restricted before production
- No authentication on endpoints — intentional for local testing

**When Azure is fixed — production security checklist:**

| Layer | Local (now) | Azure (later) |
|---|---|---|
| Auth | None | Azure AD OAuth2 / API Management keys |
| CORS | `*` | Lock to your support portal domain |
| Transport | HTTP | HTTPS via Azure App Service |
| Rate limiting | None | Azure API Management policies |
| Secrets | None needed | Azure Key Vault |

**Quick local API key (optional, to show in the project):**
You can add a simple API key check to `src/api.py` by adding
this header dependency to each endpoint:
```python
from fastapi.security.api_key import APIKeyHeader
api_key_header = APIKeyHeader(name="X-API-Key")

async def verify_key(key: str = Depends(api_key_header)):
    if key != os.environ.get("API_KEY", "dev-key"):
        raise HTTPException(status_code=403, detail="Invalid API key")
```
Then set `API_KEY=your-secret` as an environment variable before running uvicorn.

## Milestone 3 — Summary

| Deliverable | Status | Where |
|---|---|---|
| RAG chain (retrieve + generate) | Done | `src/rag_chain.py` |
| REST API (`/ask`, `/search`, `/health`) | Done | `src/api.py` |
| Interactive testing | Done | this notebook |
| End-to-end evaluation (BLEU, ROUGE) | Done | Step 4 |
| Security plan | Done | Step 6 |
| Azure deployment | Pending (Azure issue) | `src/api.py` is Azure App Service ready |

**Project folder structure so far:**
```
NHA-4-231/
├── src/
│   ├── __init__.py
│   ├── rag_chain.py         <- core RAG logic
│   └── api.py               <- FastAPI REST server
├── notebooks/
│   ├── Milestone_1_VSCode.ipynb
│   ├── Milestone_2_Local.ipynb
│   └── Milestone_3_Local.ipynb  <- this file
├── data/
│   ├── train_df.csv / val_df.csv / test_df.csv
│   └── faiss_index/         <- FAISS index + embeddings from M2
└── venv/
```

---
**Next -> Milestone 4:** MLflow experiment tracking, monitoring dashboard,
and automated retraining pipeline.